In [9]:
import numpy as np
import pygame
import sys


# --- Running pygame windows from Jupyter (Linux, macOS, Windows) ----------------------------
# Every cell that starts with %%pygame_window runs in its own short-lived Python process.
# Closing the window (X button) simply ends that process, so the Jupyter kernel never touches
# the window system itself. Opening/closing SDL windows inside the kernel leaves frozen
# windows behind on macOS and can hang or crash the kernel.
# Definitions from previously executed cells (classes, functions, variables) are available
# in the window cell, because those cells are replayed (silently) in the new process first.
# Variables created inside a %%pygame_window cell only exist while its window is open, so put
# anything later cells need into a normal cell.
# Close a window with its X button or Esc.
import os, json, subprocess, tempfile

_PYGAME_CHILD = r'''
import contextlib, io, json, linecache, sys, traceback
with open(sys.argv[1], encoding="utf-8") as f:
    payload = json.load(f)
ns = {"__name__": "__main__"}
for src in payload["setup"]:  # replay earlier notebook cells, ignore their output and errors
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            exec(compile(src, "<notebook cell>", "exec"), ns)
    except BaseException:
        pass

# Every window opens in front with keyboard focus, and Esc closes it like the X button
# (Esc is turned into a normal pygame.QUIT event, handled by the cell's own QUIT check).
import pygame
_orig_set_mode, _orig_get = pygame.display.set_mode, pygame.event.get

def _set_mode(*args, **kwargs):
    surface = _orig_set_mode(*args, **kwargs)
    try:  # bring the new window to the front with keyboard focus
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if hasattr(pygame, "Window"):  # pygame-ce
                pygame.Window.from_display_module().focus()
            else:  # classic pygame
                from pygame._sdl2.video import Window
                Window.from_display_module().focus()
    except Exception:
        pass
    return surface

def _get(*args, **kwargs):
    return [pygame.event.Event(pygame.QUIT) if e.type == pygame.KEYDOWN and e.key == pygame.K_ESCAPE else e
            for e in _orig_get(*args, **kwargs)]

pygame.display.set_mode, pygame.event.get = _set_mode, _get

main = "\n" + payload["main"]  # leading newline: line numbers match the notebook cell (line 1 = %%pygame_window)
linecache.cache["<pygame cell>"] = (len(main), None, main.splitlines(True), "<pygame cell>")
try:
    exec(compile(main, "<pygame cell>", "exec"), ns)
except SystemExit:
    raise
except BaseException as e:
    traceback.print_exception(type(e), e, e.__traceback__.tb_next)
    sys.exit(1)
'''

_OPENS_WINDOW = "display." + "set_mode("  # split so this cell does not match itself

def _replayable_cells():
    """Source of previously executed cells, without magics and without cells that open windows."""
    cells = []
    for src in get_ipython().history_manager.input_hist_raw:
        if not src.strip() or src.lstrip().startswith("%%") or _OPENS_WINDOW in src:
            continue
        lines = [l for l in src.splitlines() if not l.lstrip().startswith(("%", "!"))]
        cells.append("\n".join(lines))
    return cells

class PygameWindowError(Exception):
    pass

def pygame_window(line, cell):
    with tempfile.TemporaryDirectory() as tmp:
        payload_file = os.path.join(tmp, "payload.json")
        child_file = os.path.join(tmp, "child.py")
        with open(payload_file, "w", encoding="utf-8") as f:
            json.dump({"setup": _replayable_cells(), "main": cell}, f)
        with open(child_file, "w", encoding="utf-8") as f:
            f.write(_PYGAME_CHILD)
        env = dict(os.environ, PYGAME_HIDE_SUPPORT_PROMPT="1", PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
        proc = subprocess.Popen([sys.executable, child_file, payload_file], cwd=os.getcwd(), env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding="utf-8", errors="replace")
        try:
            for out_line in proc.stdout:  # forward prints and tracebacks to the notebook
                print(out_line, end="")
            proc.wait()
        except KeyboardInterrupt:  # "Interrupt" button closes the window
            proc.terminate()
            proc.wait()
            print("pygame window closed by interrupt")
            return
    if proc.returncode != 0:
        raise PygameWindowError("the pygame cell raised an error, see the traceback above")

def _short_error(shell, etype, evalue, tb, tb_offset=None):
    print(f"{etype.__name__}: {evalue}", file=sys.stderr)  # the real traceback is printed above

try:
    get_ipython().register_magic_function(pygame_window, "cell", "pygame_window")
    get_ipython().set_custom_exc((PygameWindowError,), _short_error)
except NameError:
    pass  # not running inside IPython / Jupyter


def close_pygame():
    pygame.quit()

## Exercise 1: 2D Transformations as Affine Matrices

*Transformations between coordinate frames play an important role in robotics. 
As background for the exercises on this sheet, please refer to the linear algebra slides on affine transformations and transformation combination.*

*Considering a robot moving on a plane, its pose w.r.t. a global coordinate frame is commonly written as $\mathbf{x} = (x, y, \theta)^T$ , where $(x, y)$ denotes its position in the $xy$- plane and $\theta$ its orientation. 
The homogeneous transformation matrix that represents a pose $\mathbf{x} = (x, y, \theta)^T$ w.r.t. to the origin $(0, 0, 0)^T$ of the global coordinate system is given by*
\begin{align*}
	 \mathbf{T} = \begin{pmatrix}
		\mathbf{R}(\theta) & \mathbf{t} \\
		0 & 1
	\end{pmatrix},
	 \mathbf{R}(\theta) = \begin{pmatrix}
		\cos(\theta) & -\sin(\theta) \\
		\sin(\theta) & \cos(\theta)
	\end{pmatrix},
\mathbf{t} = \begin{pmatrix}
	x \\
	y
\end{pmatrix}
\end{align*}



(a) *Write a class in Python for a point $\mathbf{t}$ in the xy-plane for both cartesian and homogeneous coordinates*.

In [ ]:
class Point:
    """A point (x, y) in the xy-plane in cartesian coordinates."""

    def __init__(self, x, y):
        """Create a point.
        Args:
            x: x coordinate (int or float), store it as float in the attribute self.x
            y: y coordinate (int or float), store it as float in the attribute self.y
        Note:
            The provided pygame code reads the coordinates via point.x and point.y,
            so keep exactly these attribute names.
        """
        #if isinstance()
        self.x = float(x)
        self.y = float(y)

    def __add__(self, other):
        """Componentwise sum, i.e. p1 + p2.
        Args:
            other: a Point
        Returns:
            A new Point (x1 + x2, y1 + y2). Return NotImplemented if other is not a Point.
        """
        if isinstance(other, Point):
            newX = self.x + other.x
            newY = self.y + other.y
            return Point(newX,newY)
        else:
            return NotImplemented

    def __sub__(self, other):
        """Componentwise difference, i.e. p1 - p2.
        Args:
            other: a Point
        Returns:
            A new Point (x1 - x2, y1 - y2). Return NotImplemented if other is not a Point.
        """
        if isinstance(other, Point):
            newX = self.x - other.x
            newY = self.y - other.y
            return Point(newX,newY)
        else:
            return NotImplemented

    def __mul__(self, scalar):
        """Scalar multiplication from the right, i.e. p * k.
        Args:
            scalar: an int or float k (no dot product here)
        Returns:
            A new Point (k * x, k * y). Return NotImplemented if scalar is not an int or float.
        """
        if isinstance(scalar, (int, float)):
            newX = self.x * scalar
            newY = self.y * scalar
            return Point(newX,newY)
        else:
            return NotImplemented


    def __rmul__(self, scalar):
        """Scalar multiplication from the left, i.e. k * p.
        Args:
            scalar: an int or float k
        Returns:
            The same result as p * k (you can reuse __mul__).
        """
        if isinstance(scalar, (int, float)): # hat das Ding einen Sinn? - außer vertauschte Faktoren
            newX = scalar * self.x
            newY = scalar * self.y
            return Point(newX,newY)
        else:
            return NotImplemented

    def __neg__(self):
        """Negation, i.e. -p.
        Returns:
            A new Point (-x, -y).
        """
        newX = -self.x
        newY = -self.y
        return Point(newX,newY)

    def as_array(self):
        """Convert the point to a numpy array.
        Returns:
            np.array of shape (2,): [x, y]
        """
        array = np.array((self.x,self.y))
        return array


class Point_h(Point):
    """A point in homogeneous coordinates (x, y, 1)."""

    def __init__(self, x, y):
        """Create a homogeneous point.
        Args:
            x: x coordinate, stored in self.x (reuse the Point constructor)
            y: y coordinate, stored in self.y
        Note:
            Additionally store the homogeneous coordinate self.z = 1.
        """
        super().__init__(x,y)
        self.z = 1

    def as_array(self):
        """Convert the point to a numpy array in homogeneous coordinates.
        Returns:
            np.array of shape (3,): [x, y, 1]
        """
        array = np.array((self.x,self.y,self.z))
        return array

(b) *Next, write a class in Python for a 2D rotation matrix that either creates matrix $R(\theta)$ from a given $2\times2$ matrix or from an angle $\theta$*. *In case a matrix is given also check if it is orthogonal and has a determinant of $+1$*.

In [ ]:
class Rotation:
    """A 2D rotation matrix R(theta)."""

    def __init__(self, matrix):
        """Create instance from a given rotation matrix.
        Args:
            matrix: array-like of shape (2x2), convert it to a float np.array
        Raises:
            ValueError: if the shape is not (2x2), if the matrix is not orthogonal (R^T R = I)
                or if its determinant is not +1. Compare floats with a tolerance
                (np.allclose, np.isclose), never with ==.
        Note:
            Store the matrix in the attribute self.matrix, later code relies on this name.
        """
        # Check properties
        self.matrix = np.array(matrix, dtype = float)

        if self.matrix.shape != (2,2):
            raise ValueError("Matrix hat keine 2x2-Form.")

        det = np.linalg.det(self.matrix)
            
        if not np.isclose(det, 1.0, rtol =  1e-5, atol =  1e-8):
            raise ValueError("Die Determinante liegt nicht nahe bei 1.")
        
        identity = self.matrix.T @ self.matrix
                
        if not np.allclose(identity, np.eye(2)):
            raise ValueError("Die Matrix ist icht orthogonal.")     

    @classmethod
    def from_angle(cls, theta):
        """Create instance from a rotation angle.
        Args:
            theta: rotation angle in radians (counter-clockwise)
        Returns:
            A Rotation instance holding R(theta) = [[cos, -sin], [sin, cos]]
        """
        # Theta is given in radians
        # R = [[[np.cos(np.deg2rad(theta)), -(np.sin)(np.deg2rad(theta))], [np.sin(np.deg2rad(theta)), np.cos(np.deg2rad(theta))]]]
        # braucht man nicht, da np.cos() und np.sin() standardmäßig Radiant erwarten
        R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
        return cls(R)

    def get_angle(self):
        """Recover the rotation angle from the matrix (given, nothing to implement).
        Returns:
            The angle theta in radians in (-pi, pi], computed as atan2(sin(theta), cos(theta))
            = atan2(R[1, 0], R[0, 0]).
        """
        # In radians
        return np.arctan2(self.matrix[1][0], self.matrix[0][0])

    def apply(self, point):
        """Rotate a single point about the origin, i.e. p' = R p.
        Args:
            point: a Point
        Returns:
            A new rotated Point.
        Raises:
            TypeError: if point is not a Point
        """
        if isinstance(point, Point):
            #newX = point.x * np.cos(self.matrix) + point.y * -np.sin(self.matrix)
            #newY = point.x * np.sin(self.matrix) + point.y * np.cos(self.matrix)
            #return Point(newX,newY)
            newP = self.matrix @ point.as_array()
            return Point(newP[0], newP[1])
        else:
            raise TypeError("Punkt ist kein Punkt")

    def apply_to_array(self, points):
        """Rotate several points at once.
        Args:
            points: an iterable (list, tuple, ...) of Point objects
        Returns:
            A list of new rotated Points in the same order.
        Raises:
            TypeError: if any element is not a Point
        Hint:
            Stack the points into an (N x 2) array of row vectors, then rotate all of them
            with one matrix product instead of a Python loop (mind the transpose).
        """
        # Points saved as row vectors (p^T) so R*p = p^T*R^T is used
        for point in points:
            if not isinstance(point, Point):
                raise TypeError("Einer der Punkte ist kein Punkt")

        array = np.array([point.as_array() for point in points])
        rotation = array @ self.matrix.T
        result = []
        for i in range(len(rotation)):
            result.append(Point(rotation[i][0], rotation[i][1]))
        return result

    def inverse(self):
        """Invert the rotation.
        Returns:
            A new Rotation representing R^-1 (for rotation matrices R^-1 = R^T).
        """
        inverse = self.matrix.T
        return Rotation(inverse)

    def __matmul__(self, other):
        """Concatenate two rotations, i.e. R1 @ R2.
        Args:
            other: a Rotation which is applied first
        Returns:
            A new Rotation with matrix R1 R2 (applies R2, then R1).
        Raises:
            TypeError: if other is not a Rotation
        """
        # (R1 @ R2) applies R2, then R1
        pass

(c) *Finally, write a class in Python for affine transformations using the Rotation and Point Python classes which provides the specified methods.*

In [4]:
class Transformation:
    """A 2D rigid (affine) transformation T = [[R, t], [0, 0, 1]] in homogeneous coordinates.

    Naming convention used in this sheet: a2b maps points from frame a into frame b,
    i.e. robot2world corresponds to the slide notation {}^W T_R.
    """

    def __init__(self, rotation=None, translation=None):
        """Create instance from a rotation and a translation.
        Args:
            rotation: a Rotation instance, defaults to the identity rotation if None
            translation: a Point (t_x, t_y), defaults to Point(0, 0) if None
        Raises:
            TypeError: if rotation is not a Rotation or translation is not a Point
        Note:
            Transformation() without arguments must be the identity transformation.
            Provide the attributes
                self.matrix:      np.array of shape (3x3), the homogeneous matrix T
                self.rotation:    the Rotation part R
                self.translation: the translation part t as a Point
            The provided pygame code and create_ellipse() rely on these names.
        """
        pass

    def _set_matrix(self, matrix):
        """Helper (suggested): validate a homogeneous matrix and set all attributes from it.
        Args:
            matrix: array-like of shape (3x3)
        Raises:
            ValueError: if the shape is not (3x3), the last row is not [0, 0, 1] or the
                upper-left (2x2) block is not a valid rotation (orthogonal, det = +1)
        Note:
            Sets self.matrix, self.rotation and self.translation, so __init__ and
            from_matrix can share the same code.
        """
        pass

    @classmethod
    def from_matrix(cls, matrix):
        """Create instance from a given transformation matrix.
        Args:
            matrix: array of a (3x3) homogenous transformation matrix
        Returns:
            A Transformation instance with the same attributes as one built via __init__.
        """
        pass

    def apply(self, point):
        """Apply transformation to a point.
        Args:
            point: either a Point or Point_h object
        Returns: 
            The transformed Point or Point_h depending on which was input.
        Raises:
            TypeError: if point is not a Point or Point_h
        Hint:
            Write the point in homogeneous coordinates [x, y, 1] and multiply with self.matrix.
        """
        pass

    def compose(self, other):
        """Compose two transformation matrices to one transformation.
        Args:
            other: the second transformation which is applied first, i.e. lsr2world = rob2world.compose(lsr2rob)
        Returns:
            The composed Transformation instance
        Raises:
            TypeError: if other is not a Transformation
        """
        pass

    def inverse(self):
        """Inverts a Transformation.
        Returns:
            The new inverted Transformation, i.e. world2robot = robot2world.inverse()
        Hint:
            Use the structure of T instead of a general matrix inverse:
            T^-1 = [[R^T, -R^T t], [0, 0, 1]]
        """
        pass

## Exercise 2: Matrices and Ellipses

*Consider the matrices*

\begin{align*}
    \mathbf{A} = \begin{pmatrix}
        0.25 & 0.1 \\
        0.2 & 0.5
    \end{pmatrix}, \
    \mathbf{B}= \begin{pmatrix}
        0.25 & -0.3 \\
        -0.3 & 0.5
\end{pmatrix}
\end{align*}

*In robotics, the uncertainty of a robots state can be represented as a covariance matrix so we would like to now if these matrices are valid covariance matrices, i. e. if they are symmetric positive semidefinite.*
- (a) *Are they symmetric positive semidefinite? Write a short python program.*

In [5]:
def is_mat_pos_semidefinite(mat):
    """Check whether a matrix is a valid covariance matrix (symmetric positive semidefinite).
    Args:
        mat: np.array of arbitrary shape
    Returns:
        True if all of the following properties hold, otherwise False:
            1. mat is 2-dimensional (mat.ndim == 2)
            2. mat is square (n x n)
            3. mat is symmetric, i.e. mat == mat^T (Cov(x_i, x_j) = Cov(x_j, x_i))
            4. all eigenvalues of mat are >= 0, i.e. a^T mat a = Var(a^T x) >= 0 for every a
    Hint:
        Check the properties in this order, the eigenvalue test only makes sense for square,
        symmetric matrices. Floats are rarely exactly equal, so compare with a tolerance
        (np.allclose). np.linalg.eigvalsh computes the (real) eigenvalues of a symmetric
        matrix. Round-off can produce tiny negative eigenvalues that should count as 0.
    """
    pass


# Define the matrices A and B from the task as np.arrays
A = np.array([[0.25, 0.1],
              [0.2,  0.5]])
B = np.array([[0.25, -0.3],
              [-0.3,  0.5]])

# Check both matrices with your function and print the results.
# Which of the properties above fails for a matrix that is not a valid covariance matrix?
pass

- (b) *Write a python function which computes the points of the uncertainty ellipse from a given robot state and covariance. Only use uncertainty in position ignoring orientation meaning only  ```[:2, :2]``` of the covariance. The function will get passed a Transformation instance which represents a robot state and a covariance matrix, and returns a list of Points in world coordinates that the provided pygame code draws as a closed line.*

In [6]:
def create_ellipse(state, cov, nsig=2):
    """Creates an ellipse from robot state and covariance.
    Args:
        state: robot pose as a Transformation instance, the ellipse is centered at state.translation
        cov: covariance matrix, np.array of arbitrary shape, only cov[:2, :2] (position) is used
        nsig: number of standard deviations k, scales the principal axes of the ellipse
    Returns:
        list of 80 Points in world coordinates sampled evenly along the k-sigma ellipse
    Hint:
        p(phi) = mu + V k Lambda^(1/2) [cos(phi), sin(phi)]^T with Sigma = V Lambda V^T.
        The semi-axes have length k*sqrt(lambda) (standard deviations, not variances), clamp
        tiny negative eigenvalues to 0 before the square root, and do not rotate by the robot
        heading because the covariance is already given in world coordinates.
    """
    # Eigen-decomposition of the (2x2) position block of the covariance
    # Reorder from largest to smallest
    # Clamp tiny negative values due to floating point rounding
    # For 80 evenly spaced angles phi on the unit circle:
    #     Covariance is in world coordinates: do not rotate by robot heading
    #     Scale unit circle to ellipse, sqrt of the eigenvalues scales towards principal direction in length of requested standard deviation
    #     and eigenvectors @ rotates point toward covariance orientation
    #     Shift the result to the robot position (center) and store it as a Point
    pass

*Use the following code to check your ellipse.*

In [7]:
# Robot state and covariance used by the following pygame windows
robot_state = [0.0, 0.0, 0.0]
covariance = np.array([
    [0.04,   0.012, 0.0],
    [0.012,  0.0225, 0.0],
    [0.0,    0.0,    np.radians(5) ** 2],
])

In [8]:
%%pygame_window
WIDTH, HEIGHT = 600, 780
FPS = 20

BG = (24, 29, 39)
PANEL = (32, 39, 52)
GRID = (54, 64, 80)
WHITE = (235, 239, 245)
MUTED = (160, 172, 190)
BLUE = (70, 145, 255)
CYAN = (70, 220, 235)
ORANGE = (255, 170, 65)
GREEN = (75, 220, 140)

WORLD_SCALE = 45 # How many pixels represent a robot step on coordinate axes

pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Laser Measurements Coordinate Frames")

clock = pygame.time.Clock()
font = pygame.font.SysFont("consolas", 16)

world_view = pygame.Rect(20, 65, 560, 560)

def world_to_screen(point):
    return (
        round(world_view.centerx + point.x * WORLD_SCALE),
        round(world_view.centery - point.y * WORLD_SCALE), # Subtract because pygame y axis points downward
    )

def draw_grid(view, scale):
    for x in range(-20, 21):
        px = view.centerx + x * scale
        if view.left <= px <= view.right:
            color = MUTED if x == 0 else GRID
            pygame.draw.line(screen, color, (px, view.top), (px, view.bottom), 2 if x == 0 else 1)

    for y in range(-20, 21):
        py = view.centery + y * scale
        if view.top <= py <= view.bottom:
            color = MUTED if y == 0 else GRID
            pygame.draw.line(screen, color,(view.left, py), (view.right, py), 2 if y == 0 else 1)

try:
    running = True

    while running:
        clock.tick(FPS)

        screen.fill(BG)

        for event in pygame.event.get():
    
                if event.type == pygame.QUIT:
                    running = False

        pygame.draw.rect(screen, PANEL, world_view, border_radius=8)  
        old_clip = screen.get_clip()
    
        screen.set_clip(world_view)
        draw_grid(world_view, WORLD_SCALE)       
        robot_pos = world_to_screen(Point(robot_state[0], robot_state[1]))
        pygame.draw.circle(screen, BLUE, robot_pos, 10)
        pygame.draw.circle(screen, WHITE, robot_pos, 10, 2)
        points = create_ellipse(Transformation(), covariance) # Pass state in robot frame, not pixels
        screen_points = []
        # Transform from robot units to pygame pixels
        for point in points:
            screen_points.append(world_to_screen(point))
        pygame.draw.lines(screen, ORANGE, True, screen_points, 2)
        pygame.display.flip()
finally:
    close_pygame()

Traceback (most recent call last):
  File "<pygame cell>", line 63, in <module>
    robot_pos = world_to_screen(Point(robot_state[0], robot_state[1]))
  File "<pygame cell>", line 28, in world_to_screen
    round(world_view.centerx + point.x * WORLD_SCALE),
                               ^^^^^^^
AttributeError: 'Point' object has no attribute 'x'


PygameWindowError: the pygame cell raised an error, see the traceback above


## Exercise 3: Sensing

*A robot is at the origin and not rotated, i.e., $\mathbf{x} = (0, 0, 0)^T$, in the world's frame of reference and has a laser range finder mounted on at $x = 0.2m$, $y = 0.0m$, $\theta = \pi$ with respect to the robot’s frame of reference (All angles in radians).*
*The following code creates three views which show some landmarks in world reference fame, robot reference frame and laser reference frame using the classes and functions from exercises $1$ and $2$.*

*For the code to work you need to add the necessary transformations at the marked lines, i.e. replace every `pass` by your code. Until then the cell does not run.*

*In the resulting pygame window you can place the robot by clicking on the world coordinate system and rotate the robot by using the arrow keys or the slider.*

In [ ]:
%%pygame_window
WIDTH, HEIGHT = 1800, 780
FPS = 20
MIN_DEG, MAX_DEG = -180, 180

# Color settings
BG = (24, 29, 39)
PANEL = (32, 39, 52)
GRID = (54, 64, 80)
WHITE = (235, 239, 245)
MUTED = (160, 172, 190)
BLUE = (70, 145, 255)
CYAN = (70, 220, 235)
ORANGE = (255, 170, 65)
GREEN = (75, 220, 140)

# Initial robot pose in the world
# robot_x, robot_y: initial position of the robot in the world frame (the robot starts at the origin)
# theta: initial heading of the robot in radians
# robot2world: Transformation from the robot frame into the world frame ({}^W T_R), built from robot_x, robot_y, theta
# laser2robot: Transformation from the laser frame into the robot frame ({}^R T_S),
#              the laser is mounted at x = 0.2 m, y = 0.0 m, theta = pi w.r.t. the robot
robot_x = pass
robot_y = pass
theta = pass
robot2world = pass
laser2robot = pass


world_landmarks = [
    Point(1.0, 1.0),
    Point(3.0, -3.0),
    Point(5.0, 1.0),
    Point(1.0, 3.0),
    Point(5.0, -4.0),
    Point(-2.0, -1.0),
    Point(1.0, 5.0),
    Point(-6.0, -1.0),
    Point(-3.0, -5.0),
]


# Window resize helpers
def layout_views(width, height):
    global world_view, robot_view, laser_view, slider

    margin = 20
    gap = 20
    top = 65
    bottom_area = 150

    view_size = min((width - 2 * margin - 2 * gap) // 3, height - top - bottom_area)
    view_size = max(100, view_size)

    total_width = 3 * view_size + 2 * gap
    left = (width - total_width) // 2

    world_view = pygame.Rect(left, top, view_size, view_size)
    robot_view = pygame.Rect(left + view_size + gap, top, view_size, view_size)
    laser_view = pygame.Rect(left + 2 * (view_size + gap), top, view_size, view_size)

    slider_width = min(700, width - 2 * margin)
    slider = pygame.Rect((width - slider_width) // 2, height - 80, slider_width, 8)

def view_scale(view, world_span=12, margin=0.9):
    return min(view.width, view.height) * margin / world_span

pygame.init()

screen = pygame.display.set_mode((WIDTH, HEIGHT), pygame.RESIZABLE)
world_view = pygame.Rect(20, 65, 560, 560)
robot_view = pygame.Rect(620, 65, 560, 560)
laser_view = pygame.Rect(1220, 65, 560, 560)
layout_views(*screen.get_size())
pygame.display.set_caption("Laser Measurements Coordinate Frames")

clock = pygame.time.Clock()
font = pygame.font.SysFont("consolas", 16)
small_font = pygame.font.SysFont("consolas", 13)
title_font = pygame.font.SysFont("consolas", 21, bold=True)

def update_robot_pose(x, y, angle):
    global robot_x, robot_y, theta, robot2world

    robot_x = x
    robot_y = y
    theta = max(np.radians(MIN_DEG),min(angle, np.radians(MAX_DEG))) # Clamp angle

    robot2world = Transformation(rotation=Rotation.from_angle(theta),translation=Point(robot_x, robot_y))

# Coordinate transformations between robot units to pygame pixels, no robot coordinate frame transformations
def world_to_screen(point):
    scale = view_scale(world_view)
    return (
        round(world_view.centerx + point.x * scale),
        round(world_view.centery - point.y * scale))# Subtract because pygame y axis points downward

def robot_to_screen(point):
    scale = view_scale(robot_view)
    return (
        round(robot_view.centerx + point.x * scale),
        round(robot_view.centery - point.y * scale))

def laser_to_screen(point):
    scale = view_scale(laser_view)
    return (
        round(laser_view.centerx + point.x * scale),
        round(laser_view.centery - point.y * scale))

def screen_to_world(pos):
    scale = view_scale(world_view)
    return ((pos[0] - world_view.centerx) / scale, (world_view.centery - pos[1]) / scale)

# Drawing helpers
def draw_text(text, pos, color=WHITE, font_obj=font):
    surface = font_obj.render(text, True, color)
    screen.blit(surface, pos)


def draw_grid(view, scale):
    half_x = int(np.ceil((view.width / 2) / scale))
    half_y = int(np.ceil((view.height / 2) / scale))

    for x in range(-half_x, half_x + 1):
        px = round(view.centerx + x * scale)
        color = MUTED if x == 0 else GRID
        pygame.draw.line(screen, color,(px, view.top), (px, view.bottom), 2 if x == 0 else 1)

    for y in range(-half_y, half_y + 1):
        py = round(view.centery + y * scale)
        color = MUTED if y == 0 else GRID
        pygame.draw.line(screen, color, (view.left, py), (view.right, py),2 if y == 0 else 1)

def draw_axes(converter, transform):
    # Draw the robot coordinate axes by transforming with robot poses

    origin = transform.apply(Point(0, 0))
    x_end = transform.apply(Point(0.8, 0))
    y_end = transform.apply(Point(0, 0.8))

    p0 = converter(origin)
    px = converter(x_end)
    py = converter(y_end)

    pygame.draw.line(screen, CYAN, p0, px, 3)
    pygame.draw.line(screen, GREEN, p0, py, 3)

    draw_text("X", (px[0] + 4, px[1]), CYAN, small_font)
    draw_text("Y", (py[0] + 4, py[1]), GREEN, small_font)


def draw_robot():
    # Robot origin in the world
    # origin: the point (0, 0) of the robot frame expressed in the world frame, as a Point
    origin = pass
    heading = robot2world.apply(Point(0.7, 0))

    p0 = world_to_screen(origin)
    p1 = world_to_screen(heading)

    pygame.draw.circle(screen, BLUE, p0, 10)
    pygame.draw.circle(screen, WHITE, p0, 10, 2)
    pygame.draw.line(screen, ORANGE, p0, p1, 4)
    pygame.draw.circle(screen, ORANGE, p1, 4)

# Angle slider
slider = pygame.Rect(250, 700, 700, 8)

dragging_slider = False

def angle_to_slider_x(degrees):
    fraction = (degrees - MIN_DEG) / (MAX_DEG - MIN_DEG)
    return round(slider.left + fraction * slider.width)


def slider_x_to_angle(x):
    fraction = (x - slider.left) / slider.width
    fraction = max(0.0, min(1.0, fraction))
    return MIN_DEG + fraction * (MAX_DEG - MIN_DEG)


# Main loop
show_axes = True
try:
    running = True

    while running:
        clock.tick(FPS)

        for event in pygame.event.get():

            if event.type == pygame.QUIT:
                running = False

            elif event.type == pygame.VIDEORESIZE:
                layout_views(*screen.get_size())  # the RESIZABLE window surface resizes itself

            elif event.type == pygame.MOUSEBUTTONDOWN:
                if event.button == 1:
                    # Change orientation
                    if slider.inflate(0, 24).collidepoint(event.pos):
                        dragging_slider = True
                        degrees = slider_x_to_angle(event.pos[0])
                        update_robot_pose(robot_x, robot_y, np.radians(degrees))
                    # Change pose 
                    elif world_view.collidepoint(event.pos):
                        x, y = screen_to_world(event.pos) # Transform from pygame pixels to robot world
                        update_robot_pose(x, y, theta)

            elif event.type == pygame.MOUSEBUTTONUP:
                if event.button == 1:
                    dragging_slider = False

            # Change orientation
            elif event.type == pygame.MOUSEMOTION:
                if dragging_slider:
                    degrees = slider_x_to_angle(event.pos[0])
                    update_robot_pose(robot_x, robot_y, np.radians(degrees))

            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_LEFT:
                    update_robot_pose(robot_x, robot_y, theta - np.radians(5))

                elif event.key == pygame.K_RIGHT:
                    update_robot_pose(robot_x, robot_y, theta + np.radians(5))

                elif event.key == pygame.K_a:
                    show_axes = not show_axes

                # Put robot back on origin
                elif event.key == pygame.K_r:
                    update_robot_pose(0, 0, 0)


        screen.fill(BG)

        window_width, window_height = screen.get_size()

        pygame.draw.rect(screen, PANEL, world_view, border_radius=8)
        pygame.draw.rect(screen, PANEL, robot_view, border_radius=8)
        pygame.draw.rect(screen, PANEL, laser_view, border_radius=8)

        draw_text("1. WORLD FRAME", (world_view.left, 25), WHITE, title_font)
        draw_text("2. ROBOT FRAME",(robot_view.left, 25), WHITE, title_font)
        draw_text("3. LASER FRAME", (laser_view.left, 25), WHITE, title_font)

        old_clip = screen.get_clip()

        # World view
        screen.set_clip(world_view)
        draw_grid(world_view, view_scale(world_view))

        if show_axes:
            draw_axes( world_to_screen, robot2world)

        # Draw fixed world landmarks as green circles
        for i, landmark in enumerate(world_landmarks):
            pos = world_to_screen(landmark)

            pygame.draw.circle(screen, GREEN, pos, 7)

            label = small_font.render(f"L{i}", True, WHITE)
            screen.blit(label, (pos[0] + 8, pos[1] - 8))

        draw_robot()
        points = create_ellipse(robot2world, covariance)
        screen_points = []
        # Transform from robot units to pygame pixels
        for point in points:
            screen_points.append(world_to_screen(point))
        pygame.draw.lines(screen, ORANGE, True, screen_points, 2)
        screen.set_clip(old_clip)

        draw_text(f"Position: ({robot_x:.2f}, {robot_y:.2f}) m", (world_view.left + 5, world_view.bottom + 8), WHITE, small_font)

        # Robot view

        screen.set_clip(robot_view)
        draw_grid(robot_view, view_scale(robot_view))

        if show_axes:
            # Robot frame is the fixed local coordinate frame, origin transformation
            draw_axes( robot_to_screen, Transformation())

        pygame.draw.circle(screen, BLUE, robot_view.center, 10)
        pygame.draw.circle(screen, WHITE, robot_view.center, 10, 2)

        screen.set_clip(old_clip)

        draw_text("Landmarks sensed in robot coordinate frame", (robot_view.left + 5, robot_view.bottom + 8), WHITE, small_font)

        # Laser view
        screen.set_clip(laser_view)
        draw_grid(laser_view, view_scale(laser_view))

        if show_axes:
            # Also fixed local coordinate frame
            draw_axes( laser_to_screen, Transformation())

        pygame.draw.circle(screen, CYAN, laser_view.center, 8)
        pygame.draw.circle(screen, WHITE, laser_view.center, 8, 2)

        screen.set_clip(old_clip)
        draw_text("Landmarks sensed in laser coordinate frame", (laser_view.left + 5, laser_view.bottom + 8), WHITE, small_font)

        # Slider and controls
        degrees = np.degrees(theta)

        #draw_text(f"Heading: {degrees:.1f} deg",(20, 680), WHITE)
        draw_text(f"Heading: {degrees:.1f} deg", (20, window_height - 100), WHITE)

        pygame.draw.line(screen, MUTED,(slider.left, slider.centery), (slider.right, slider.centery), 4)

        for d in (-180, -90, 0, 90, 180):
            x = angle_to_slider_x(d)
            pygame.draw.line(screen, WHITE,(x, slider.top - 5),(x, slider.bottom + 5), 2)
            draw_text(str(d), (x - 12, slider.bottom + 8), MUTED, small_font)

        pygame.draw.circle(screen, ORANGE, (angle_to_slider_x(degrees), slider.centery), 9)

        draw_text("Click world view: place robot | Drag slider: rotate | Arrows: rotate | A: axes | R: reset", (20, window_height - 30), MUTED, small_font)
    
        # Transform landmarks
        # landmarks_robot: list of all world_landmarks expressed in the robot frame (list of Points, same order)
        # landmarks_laser: list of all world_landmarks expressed in the laser frame (list of Points, same order)
        # Hint: you are given robot2world and laser2robot, but here you need the opposite directions,
        #       p_S = (R_T_S)^-1 (W_T_R)^-1 p_W (see slide "Homogeneous Coordinates")
        landmarks_robot = pass # Transform landmarks to the robot reference frame
        landmarks_laser = pass # Transform landmarks to the laser reference frame

        # Robot origin in laser frame
        # robot_origin_laser: the point (0, 0) of the robot frame expressed in the laser frame, as a Point
        robot_origin_laser = pass
        robot_origin_pos = laser_to_screen(robot_origin_laser)
        pygame.draw.circle(screen, BLUE, robot_origin_pos, 7)
        pygame.draw.circle(screen, WHITE, robot_origin_pos, 7, 2)

        # Draw landmarks
        for i, landmark in enumerate(landmarks_robot):
            pos = robot_to_screen(landmark) # Transform from robot units to pygame pixels
            pygame.draw.circle(screen, GREEN, pos, 7)
            label = small_font.render( f"L{i}", True, WHITE)
            screen.blit(label, (pos[0] + 8, pos[1] - 8))

        for i, landmark in enumerate(landmarks_laser):
            pos = laser_to_screen(landmark)
            pygame.draw.circle(screen, GREEN, pos, 7)
            label = small_font.render(f"L{i}", True, WHITE)
            screen.blit(label, (pos[0] + 8, pos[1] - 8))

        pygame.display.flip()
finally:
    close_pygame()